# Lev encoder decision model — Kaggle smoke run

Trains a **ModernBERT** backbone with a two-way `false / true` head on a frozen
Lev pack, then scores it with the unchanged `lev report`.

## Before you run

1. **Settings → Internet → On.** The backbone is pulled from the Hub by commit SHA.
2. **Settings → Accelerator → GPU T4 x2** (or P100). One card is enough; the
   second is unused because this path is single-device.
3. **The frozen pack is already attached** as the `saltbo/lev-en-joint-002`
   Dataset, mounted at `/kaggle/input/lev-en-joint-002`. If you swap packs,
   update `DATASET` below and add the new source to the kernel metadata. The
   pack must contain `manifest.json`, `rubric.json`, `records.jsonl` and the
   four split files; the first cell refuses anything else.

## What this does and does not prove

It proves the encoder path trains, reloads and scores on this hardware, and what
the development accuracy and wall-clock actually are. It does **not** replace a
test-set reading, an independent calibration, or a release decision: pick and
freeze the model on development, then read test once.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True,
).stdout if shutil.which("nvidia-smi") else
      "no nvidia-smi on PATH: this session has no GPU accelerator")

In [ ]:
REPO = "https://github.com/eslsoft/lev.git"
BRANCH = "experiment/plan-a-modernbert"
CONFIG = "configs/encoder-modernbert-base-en-joint-010.toml"

WORK = Path("/kaggle/working")
SRC = WORK / "lev"
HINT = Path("/kaggle/input/lev-en-joint-010")

required = {"manifest.json", "rubric.json", "records.jsonl"} | {
    f"{split}.jsonl" for split in ("train", "development", "calibration", "test")
}


def is_pack(path):
    return path.is_dir() and required <= {p.name for p in path.glob("*")}


def find_pack(root):
    # Kaggle now mounts datasets under /kaggle/input/datasets/<owner>/<slug>,
    # so discover the pack instead of pinning a layout that can move again.
    if is_pack(root):
        return root
    for candidate in sorted(p for p in root.rglob("*") if p.is_dir()):
        if is_pack(candidate):
            return candidate
    return None


DATASET = HINT if is_pack(HINT) else find_pack(Path("/kaggle/input"))
if DATASET is None:
    seen = [
        str(p.relative_to("/kaggle/input"))
        for p in Path("/kaggle/input").rglob("*")
    ][:40]
    raise SystemExit(
        f"no frozen pack under /kaggle/input; need {sorted(required)}.\n"
        f"found: {seen}\n"
        "A dataset still processing mounts as an empty directory; wait for "
        "`kaggle datasets status` to report ready, then re-run."
    )
print("frozen pack:", DATASET, "|", len(list(DATASET.glob("*"))), "files")
print("output:", WORK)
DEV_RUN = "enc-dev-005"


In [ ]:
import socket

# Kaggle gates Internet and GPU behind phone verification. Fail with the fix
# instead of a bare CalledProcessError from git clone.
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError as error:
    raise SystemExit(
        f"no outbound network: {error}\n"
        "Kaggle locks Internet and GPU behind phone verification. "
        "Open kaggle.com/settings/account -> Phone verification, then re-run."
    )

if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)], check=True
)

# --no-deps keeps Kaggle's own torch; only pull the extra if the import fails.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(SRC)],
    check=True,
)

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(SRC / "src")
# Bucketed batches keep padding small; this keeps the allocator from fragmenting.
ENV.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def lev(*args):
    print("$ lev", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", "lev", *args], cwd=SRC, env=ENV, check=True)


lev("check-upstream", "--help")  # cheap import check for the CLI itself

try:
    import torch
    import transformers
    if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (4, 48):
        raise ImportError(f"transformers {transformers.__version__} predates ModernBERT")
except ImportError as error:
    print(f"installing the encoder extra: {error}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"{SRC}[encoder]"], check=True
    )
    import torch
    import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device. Set Settings -> Accelerator -> GPU T4 x2 on this notebook "
        "and re-run; training 15,515 pairs on CPU would take hours."
    )
if torch.cuda.is_available():
    # T4 / P100 are sm_75 / sm_60: no flash kernel, so long sequences cost real memory.
    print(
        "sdpa flash", torch.backends.cuda.flash_sdp_enabled(),
        "| mem_efficient", torch.backends.cuda.mem_efficient_sdp_enabled(),
        "| math", torch.backends.cuda.math_sdp_enabled(),
    )
    print("capability", torch.cuda.get_device_capability(0))

## Score development with the declared threshold

This run re-scores the saved checkpoint at the pre-registered `threshold = 0.39` from the config, so its own `job.json` records the threshold that produced its decisions.

`lev encode-benchmark` never trains and never overwrites: a re-run needs a new `DEV_RUN`.


In [ ]:
CHECKPOINT = next(
    p.parent for p in Path("/kaggle/input").rglob("model.safetensors")
)
print("checkpoint:", CHECKPOINT)

lev(
    "encode-benchmark",
    "--dataset", str(DATASET),
    "--config", CONFIG,
    "--checkpoint", str(CHECKPOINT),
    "--split", "development",
    "--out", str(WORK / DEV_RUN),
    "--device", "cuda",
    "--execute",
)

job = json.loads((WORK / DEV_RUN / "job.json").read_text())
print("job.json config.threshold:", job["config"].get("threshold"))
print("job.json config_sha256:", job["config_sha256"])


In [ ]:
lev(
    "report",
    "--benchmark", str(WORK / DEV_RUN),
    "--threshold", "0.39",
    "--out", str(WORK / DEV_RUN / "lev-report-t0.39.json"),
)

report = json.loads((WORK / DEV_RUN / "lev-report-t0.39.json").read_text())
print("threshold:", report["threshold"])
d = report["decisions"]
print(
    f"accuracy_all_requested {d['accuracy_all_requested']['rate']:.2%} "
    f"({d['accuracy_all_requested']['numerator']}/{d['accuracy_all_requested']['denominator']})"
)
print("completed", d["completed"], "missing", d["missing"])
for kind, block in report["by_kind"].items():
    a = block["accuracy_all_requested"]
    print(f"  {kind}: {a['numerator']}/{a['denominator']}")
